Imports

In [ ]:
!pip install transformers
!pip install datasets
!pip install torch
!pip install scikit-learn

Data Loading

In [ ]:
import pandas as pd

df = pd.read_csv("amazon.csv")


Data preprocessing

In [ ]:
df["text"] = df["review_title"] + " " + df["review_content"]

change rating into sentiment

In [ ]:
def sentiment(x):
    if x >= 4:
        return 2   # positive
    elif x == 3:
        return 1   # neutral
    else:
        return 0   # negative

df["rating"] = pd.to_numeric(df["rating"], errors='coerce')
df["label"] = df["rating"].apply(sentiment)

Cleaning

In [ ]:
import re

def clean_text(text):
    text = re.sub(r"http\S+", "", str(text))
    text = re.sub(r"[^a-zA-Z ]", " ", text)
    return text.lower()

df["text"] = df["text"].apply(clean_text)

data splitting

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2
)

Tokenizer

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
train_encodings = tokenizer(train_texts, truncation=True, padding=True)
val_encodings = tokenizer(val_texts, truncation=True, padding=True)

In [ ]:
import torch

class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, train_labels)
val_dataset = Dataset(val_encodings, val_labels)

BERT

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training

In [ ]:
from transformers import Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# =========================
# Metrics Function
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted'
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

# =========================
# Training Arguments
# =========================
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    max_steps=500,


    eval_strategy="steps", # Changed from evaluation_strategy
    eval_steps=50,

    logging_steps=50,
    save_steps=100,

    fp16=True
)

# =========================
# Trainer
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# =========================
# Train
# =========================
trainer.train()

# =========================
# Final Evaluation
# =========================
results = trainer.evaluate()
print(results)

Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
50,0.597292,0.593354,0.776271,0.678497,0.602597,0.776271
100,0.672910,0.541242,0.776271,0.678497,0.602597,0.776271
150,0.586884,0.522158,0.776271,0.678497,0.602597,0.776271
200,0.552691,0.566724,0.779661,0.686471,0.824986,0.779661
250,0.568342,0.500527,0.786441,0.762809,0.756316,0.786441
300,0.584244,0.588062,0.776271,0.678497,0.602597,0.776271
350,0.596711,0.543967,0.776271,0.678497,0.602597,0.776271
400,0.456771,0.566075,0.789831,0.732105,0.760453,0.789831
450,0.473164,0.570899,0.776271,0.766241,0.759260,0.776271
500,0.464147,0.597650,0.776271,0.754355,0.746407,0.776271


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.5976496338844299, 'eval_accuracy': 0.7762711864406779, 'eval_f1': 0.7543546848631595, 'eval_precision': 0.7464074443336657, 'eval_recall': 0.7762711864406779, 'eval_runtime': 2.8125, 'eval_samples_per_second': 104.888, 'eval_steps_per_second': 26.311, 'epoch': 1.694915254237288}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'eval_loss': 0.5976496338844299,
 'eval_accuracy': 0.7762711864406779,
 'eval_f1': 0.7543546848631595,
 'eval_precision': 0.7464074443336657,
 'eval_recall': 0.7762711864406779,
 'eval_runtime': 2.9096,
 'eval_samples_per_second': 101.389,
 'eval_steps_per_second': 25.433,
 'epoch': 1.694915254237288}

In [ ]:
tokenizer = tokenizer

In [ ]:
import torch

def predict_review(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_class_id = torch.argmax(logits, dim=1).item()

    return predicted_class_id

In [ ]:
review = "This product is amazing and works perfectly!"
print(predict_review(review))

2


In [ ]:
label_map = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

In [ ]:
num_labels = 3

In [ ]:
import torch.nn.functional as F

def debug_predict(text):
    model.eval()

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)

    print("Probabilities:", probs.cpu().numpy())

    return torch.argmax(probs, dim=1).item()

In [ ]:
def predict_review(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_class_id = torch.argmax(logits, dim=1).item()

    return label_map[predicted_class_id]

In [ ]:
print(predict_review("GOOD"))
print(predict_review("NOT GOOD"))

Positive
Negative


**Save the model**